In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [4]:
def classify_exercise(exam_name: str, statement: str, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_year = int(exam_name.split("_")[0])
    exam_name = (" - ".join(exam_name.split("_")[1:])).capitalize()

    # seleccionar modelo de examen en función del año
    if exam_year < 2020:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"^\d{1}\.-"

    elif exam_year < 2024:
        exam_model = "modelo_2020_2023"
        exercise_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2024+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    exercise_topics_dict = temas_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exercise_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year < 2020:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in exercise_topics_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif 2020 <= exam_year <= 2023:  # 2020 - 2023 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2024 + - key 1, 2, 3, 4A, 4B, 5
            exercise_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        nomenclature_pre_2020 = exam_year <= 2019 and exercise_number_key == "1"
        nomenclature_2020_2023 =  (2020 <= exam_year <= 2023) and exercise_number_key == "A"

        if nomenclature_pre_2020 or nomenclature_2020_2023:
            exercise_topic = "Formulación"
            exercise_types = ["Formulación"]

        elif exam_year >= 2024 and exercise_number_key == "4A":
            exercise_topic = "Formulación"
            exercise_types = ["Formulación orgánica"]

        else:

            ###### CLASIFICACIÓN #######

            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = exercise_topics_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topic = ""
            for topic in exercise_topic_options:
                
                # obtengo las palabras clave que identifican el tema
                # y los tipos de ejercicios del tema y sus respectivas keywords
                topic_identifying_keywords = keywords[topic]["keywords_tema"]
                for kw in topic_identifying_keywords:
                    if kw in statement:
                        exercise_topic = topic
                        break

                # si se ha clasificado el ejercicio deja de buscar
                if exercise_topic:
                    break

            # lista de tipos de ejercicio
            exercise_types = []

            # si se ha identificado el tema
            # lo siguiente es asignar tipo(s) de ejercicio
            if exercise_topic:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                exercise_type_keywords = keywords[exercise_topic]["tipos"]
                for t, kw_list in exercise_type_keywords.items():
                    # comprueba lista de keywords para ese tipo de ejercicio
                    # con que haga match una keyword con el enunciado
                    # clasifícalo com ese tipo de ejercicio
                    if any(kw in statement for kw in kw_list):
                        exercise_types.append(t)
            else:
                exercise_topic, exercise_types = None, [] 
    else:
        exercise_topic, exercise_types = None, []

    return (exam_year, exam_name, exercise_number, exercise_topic, exercise_types, statement)

In [5]:
# file = ROOT / "examenes_quimica/2013_quimica/2013_quimica_septiembre_reserva_a.pdf"

In [6]:
# exam_text = ext.extract_exercises_from_pdf(file)

# exam_year = int(file.name.split("_")[0])

# if exam_year < 2020:
#     exam_period = "2010-2019"
# elif exam_year <= 2024:
#     exam_period = "2020-2024"
# else:
#     exam_period = "2025+"

# exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

In [7]:
# print(exercises_list[11][1])

In [8]:
# classify_exercise(file.stem, exercises_list[11][1], quimica_keywords)

In [ ]:
# classified_exercises = []
# for file in (ROOT / "examenes_quimica_renombrados/").rglob("*.pdf"):
#     if file.is_file():
#         exam_text = ext.extract_exercises_from_pdf(file)
#         exam_year = int(file.name.split("_")[0])

#         if exam_year < 2020:
#             exam_period = "2010-2019"
#         elif exam_year <= 2024:
#             exam_period = "2020-2024"
#         else:
#             exam_period = "2025+"

#         exercises_tuples = ext.split_exercises(exam_text, exam_period=exam_period)

#         # exercises_list devuelve una tupla: (exam_option, statement)
#         for exercise_tuple in exercises_tuples:
#             try:
#                 statement = exercise_tuple[1]
#                 exam_option = exercise_tuple[0]
#                 classification = classify_exercise(file.name, statement, keywords=quimica_keywords)
#                 classification = list(classification)
#                 classification.append(exam_option)
#                 classified_exercises.append(classification)
#             except Exception as e:
#                 print(f"Error: file {file.name}. Exception: {e}")

In [56]:
df = pd.DataFrame(classified_exercises, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
df = df.explode("exercise_type", ignore_index=True)
df = df.sort_values("year")

In [59]:
df[df.topic.isna()]

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option
1129,2010,Quimica - junio - - rb.pdf,4,NaN,NaN,4.- Un litro de H₂S se encuentra en condicione...,Opción A
1135,2010,Quimica - junio - - rb.pdf,4,NaN,NaN,4.- Complete las siguientes reacciones e indiq...,Opción A
1102,2010,Quimica - sept - - rb.pdf,2,NaN,NaN,2.- Se tiene una mezcla de 10 g de hidrógeno y...,Opción A
1123,2010,Quimica - junio - - ra.pdf,4,NaN,NaN,4.- Indique los reactivos adecuados para reali...,Opción A
1164,2010,Quimica - sept - - ra.pdf,2,NaN,NaN,2.- Se tienen las siguientes cantidades de tre...,Opción A
...,...,...,...,...,...,...,...
173,2026,Quimica - junio - incompatibilidad - - suplen...,NaN,NaN,NaN,PREGUNTA 5.-\nMANTENER LIMPIA LA PISCINA: MUCH...,NaN
156,2026,Quimica - julio - titular - - ex.pdf,NaN,NaN,NaN,PREGUNTA 5.-\nDESINFECCIÓN EN EL HOGAR: EL PAP...,NaN
140,2026,Quimica - junio - incidencias - - suplente - ...,NaN,NaN,NaN,PREGUNTA 5.-\nMUCHO MÁS QUE ESPUMA\nLa higiene...,NaN
120,2026,Quimica - junio - titular - - titular.pdf,3A,NaN,NaN,"3A. En un recipiente de 2 L, en el que inicial...",NaN


In [74]:
df[df.exercise_type == "Formulación"]

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option
1132,2010,Quimica - junio - - rb.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
1140,2010,Quimica - junio - - junio.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
1092,2010,Quimica - sept - - rb.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
1147,2010,Quimica - junio - - junio.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
1184,2010,Quimica - sept - - sept.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
...,...,...,...,...,...,...,...
1588,2023,Quimica - suplente - b.pdf,A2,Formulación,Formulación,A2. Formule o nombre los siguientes compuestos...,NaN
1625,2023,Quimica - titular - a.pdf,A1,Formulación,Formulación,A1. Formule o nombre los siguientes compuestos...,NaN
1626,2023,Quimica - titular - a.pdf,A2,Formulación,Formulación,A2. Formule o nombre los siguientes compuestos...,NaN
1647,2023,Quimica - suplente - a.pdf,A2,Formulación,Formulación,A2. Formule o nombre los siguientes compuestos...,NaN


In [61]:
len(list((ROOT / "examenes_quimica_renombrados").rglob("*.pdf")))

109

In [75]:
nomenclature_classified_df = df.loc[df.exercise_type == "Formulación", ["year", "exam_name", "statement"]]

In [76]:
nomenclature_classified_df

,year,exam_name,statement
1132,2010,Quimica - junio - - rb.pdf,1.- Formule o nombre los compuestos siguientes...
1140,2010,Quimica - junio - - junio.pdf,1.- Formule o nombre los compuestos siguientes...
1092,2010,Quimica - sept - - rb.pdf,1.- Formule o nombre los compuestos siguientes...
1147,2010,Quimica - junio - - junio.pdf,1.- Formule o nombre los compuestos siguientes...
1184,2010,Quimica - sept - - sept.pdf,1.- Formule o nombre los compuestos siguientes...
...,...,...,...
1588,2023,Quimica - suplente - b.pdf,A2. Formule o nombre los siguientes compuestos...
1625,2023,Quimica - titular - a.pdf,A1. Formule o nombre los siguientes compuestos...
1626,2023,Quimica - titular - a.pdf,A2. Formule o nombre los siguientes compuestos...
1647,2023,Quimica - suplente - a.pdf,A2. Formule o nombre los siguientes compuestos...


In [83]:
def extraer_nombrar_compuestos(statement):
    start_position = statement.find("a)")
    end_position = statement.find("d)") - 1
    statement = statement[start_position:end_position]
    # unir valencia con Stock
    statement = re.sub(r"\s+(\([IVX]+\))", r"\1", statement)
    statement = statement.replace(";", "")
    
    return statement

In [78]:
nomenclature_classified_df["statement"] = nomenclature_classified_df["statement"].apply(lambda st: extraer_nombrar_compuestos(st))

In [79]:
nomenclature_classified_df

,year,exam_name,statement
1132,2010,Quimica - junio - - rb.pdf,a) Sulfato de manganeso(II) b) Hidróxido de ce...
1140,2010,Quimica - junio - - junio.pdf,a) Nitrito de plata b) Hidróxido de magnesio\n...
1092,2010,Quimica - sept - - rb.pdf,a) Dióxido de azufre b) Ácido hipobromoso\nc) ...
1147,2010,Quimica - junio - - junio.pdf,a) Peróxido de rubidio b) Hidrogenocarbonato d...
1184,2010,Quimica - sept - - sept.pdf,a) Cromato de plata b) Óxido de estaño(IV) c) ...
...,...,...,...
1588,2023,Quimica - suplente - b.pdf,a) Peróxido de bario; b) Bromuro de calcio; c)...
1625,2023,Quimica - titular - a.pdf,a) Óxido de manganeso(VII); b) Dicromato de po...
1626,2023,Quimica - titular - a.pdf,"a) Selenuro de plata; b) Ácido clórico; c) 1,3..."
1647,2023,Quimica - suplente - a.pdf,a) Hidróxido de hierro(III); b) Ácido nitroso;...


In [47]:
with open("nomenclature_statements.json", "r") as file:
    nomenclature_statements_dict = json.load(file)

In [84]:
nomenclature_emestrada_df = pd.DataFrame([
    {"year": int(year), "exam_name": exam_name, "statement": statement}
    for year, exercises in nomenclature_statements_dict.items()      # recorre cada año
    for exam_name, statement in exercises        # recorre cada [titulo, enunciado] de ese año
]).sort_values("year")

In [85]:
nomenclature_emestrada_df["statement"] = nomenclature_emestrada_df["statement"].apply(lambda st: extraer_nombrar_compuestos(st))

In [86]:
nomenclature_emestrada_df

,year,exam_name,statement
31,2000,química. 2000. reserva 3. . opción b,a) Hidróxido de hierro(III) b) Dióxido de azuf...
30,2000,química. 2000. reserva 3. . opción a,a) Fluoruro de calcio b) Óxido de\nantimonio(I...
29,2000,química. 2000. reserva 2. . opción b,a) Fluoruro de hidrógeno b) Cromato de\nmercur...
28,2000,química. 2000. reserva 2. . opción a,a) Óxido de cobre(I) b) Carbonato de sodio\nc)...
27,2000,química. 2000. reserva 1. . opción b,a) Hidruro de berilio b) Carbonato de magnesio...
...,...,...,...
257,2024,,
214,2025,,
215,2025,,
216,2025,,


In [88]:
nomenclature_classified_df.loc[nomenclature_classified_df.year == 2020]

,year,exam_name,statement
297,2020,Quimica - p - a.pdf,a) Sulfito de potasio; b) Peróxido de sodio; c...
298,2020,Quimica - p - a.pdf,a) Pentafluoruro de antimonio; b) Fosfato de c...
315,2020,Quimica - p - b.pdf,a) Hidrogenocarbonato de plata; b) Hidróxido d...
316,2020,Quimica - p - b.pdf,a) Nitrato de manganeso(II); b) Dihidruro de e...
337,2020,Quimica - p - e.pdf,a) Peróxido de hidrógeno; b) Hidrogenosulfito ...
338,2020,Quimica - p - e.pdf,a) Pentafluoruro de fósforo; b) Hidróxido de n...
360,2020,Quimica - p - d.pdf,a) Cromato de plata; b) Dióxido de estaño; c) ...
361,2020,Quimica - p - d.pdf,a) Nitruro de aluminio; b) Ácido hipocloroso; ...
377,2020,Quimica - p - c.pdf,a) Bromuro de hidrógeno; b) Óxido de plomo(IV)...
378,2020,Quimica - p - c.pdf,a) Permanganato de bario; b) Hidróxido de cesi...


In [89]:
nomenclature_emestrada_df.loc[nomenclature_emestrada_df.year == 2020].sort_values("year")

,year,exam_name,statement
109,2020,química. 2020. reserva 1.,a) Pentafluoruro de antimonio b) Fosfato de\nc...
108,2020,química. 2020. reserva 1.,a) Sulfito de potasio b) Peróxido de sodio\nc)...
111,2020,química. 2020. reserva 2.,a) Nitrato de manganeso(II) b) Dihidruro de\ne...
110,2020,química. 2020. reserva 2.,a) Hidrogenocarbonato de plata b) Hidróxido de...
107,2020,química. 2020. junio.,a) Permanganato de bario b) Hidróxido de\ncesi...
106,2020,química. 2020. junio.,a) Bromuro de hidrógeno b) Óxido de\nplomo(IV)...
115,2020,química. 2020. reserva 4.,a) Pentafluoruro de fósforo b) Hidróxido de\nn...
114,2020,química. 2020. reserva 4.,a) Peróxido de hidrógeno b) Hidrogenosulfito d...
113,2020,química. 2020. reserva 3.,a) Nitruro de aluminio b) Ácido hipocloroso\nc...
112,2020,química. 2020. reserva 3.,a) Cromato de plata b) Dióxido de estaño\nc) Á...


In [90]:
pd.merge(nomenclature_classified_df, nomenclature_emestrada_df, on=["year", "statement"])

,year,exam_name_x,statement,exam_name_y


In [81]:
nomenclature_classified_df.to_csv("nomenclature_classified.csv", index = False)

In [82]:
nomenclature_emestrada_df.to_csv("nomenclature_emestrada.csv", index = False)